In [ ]:
import torch
import math
import torch.nn as nn




vector jacobian product,this is the core mathematical idea for the back propagation

where the model learns form it's mistakes and try to make it rectified and you know we all have been through this

so it's just the rewiring of our understandings in the mathematical formulas to the neural networks

y = x @ w + b

so this is the eqn that gives the weighted sum

and after the model did gave it's predictions and we need to test the model predictions with the actual output 

and model calculates the loss and it finds back the actual changes to be done to reduce error

L -> Loss and this is what we can see at the end,more the loss number,the model was poor at the predictions and it needs the training

so we need to adjust the weights,inputs,and bias so they can come with the best predictions

dL/dx = dL/dy * dy/dx (as here the [partial derivates are used])

as dL/dy is the Loss,that we already knew and we will find the dy/dx here to find the dL/dx

as we can name the dL/dy as the grad_out ,as we will send that grad_out into the parameters 

let's derive the vector jacobian for inputs to find the grad

x->shape       =      (B,in_features)

weights->shape =      (out_features,in_features)

for a small exmaple consider the in_features = 2 and out_features=2

x = [x1 x2]

w = [ w11 w12 ]  (don't think why i didn't take the nested shape 😂️, this is for intuitive understanding)

    [ w21 w22 ]  (see the pattern here,w1 means the out neuron right,it was asking that,how much does the nueron 1 associated with me,and how much does the neuron 2 associated with me )

                 (so the weights are the w11 w12 as we are taking this in the form (out,in)  )

y = [y1 y2]   ( as (B,in) @ (in,out) => (B,out) so out is same a single dimensional but two columns

y = x @ weights.T +self.bias

so we get the [x1 x2] @ [w11  w21]

                        [w12  w22]

😂️don't ask me ,why i ignored the bias,ignore it for the calc purposes,as it will end up with being 0,when it is diff wrt to inputs and weights 

y1 = x1*w11 + x2*w12

y2 = x1*w21 + x2*w22

dL/dx = dL/dy * dy/dx

y = [y1 y2]

dy1/dx1 = w11

dy1/dx2 = w12

dy1/dx = [dy1/dx1 dy1/dx2]

dy1/dx = [w11 w12]

so same partial derivatives for these 

dy2/dx = [w21 w22]

dy/dx = [w11 w12]

        [w21 w22]

grad_out = [g1 g2]   -> we can even write them in [dL/dy1 dL/dy2] so we have taken this as the entire grad_out

dL/dx = grad_out @ self.weights

as of now we have derived for the inputs and we need to derive for the grad inputs

we will derive for the grad weights now

so the same eqns comes here too

y1 = x1*w11 + x2*w12

y2 = x1*w21 + x2*w22 

dL/dw11 = dL/dy1 * dy1/dw11

dL/dw11 = grad_out1 * x1

dL/dw12 = grad_out1 * x2

dL/dw21 = grad_out2 * x1

dL/dw22 = grad_out2 * x2

shape of the weights in the y -> (in,out) so transpose of (out,in)

dL/dw = [grad_out1*x1    grad_out1*x2]

        [grad_out2*x1    grad_out2*x2]

        [grad_out1]   @   [x1 x2]

        [grad_out2]     

dL/dw = grad_out.T @ x

to find the gradients for the bias

y1 = x@w + b1

y2 = x@w + b2

now we can find the gradients for the bias too

y1 = x1*w11 + x2*w12 +bias1

y2 = x1*w21 + x2*w22 +bias2

# as the bias1 is a constant and derivation wrt to itself is 1

dL1/db = g1*[1 0] = g1

dL2/db = g2*[0 1] = g2

dL/db = [g1 g2]

When there is a batch, the exact same bias vector b = [b1, b2] is added to every single row in the batch (broadcasting).

By multivariate calculus, if one parameter affects multiple outputs, its total gradient is the sum of gradients across all paths it influenced:

The total gradient of the loss with respect to a bias parameter is the sum of that neuron's output gradients across every sample in the batch.

In plain terms:For neuron 1 b1: Add up the incoming gradients g1 received from sample 1, sample 2, all the way through sample B.

For neuron 2 (b2): Add up the incoming gradients (g2) received from sample 1, sample 2, all the way through sample B.

Because the exact same bias is added to every single training example in the batch, each example has an opinion on how that bias should change. 

The total gradient is simply the sum of all those individual opinions.

😭️i literally can't be able to write this long sentences,so i have taken the help of gemini to frame these words below to make u feel more intuitive

In PyTorch, the total Loss across a batch of size B is the sum (or mean) of the individual losses:

Total Loss = Loss_1 + Loss_2 + ... + Loss_B

Now look at your forward pass for neuron 1:

    Sample 1: y1_sample1 = (x1 @ w) + b1

    Sample 2: y1_sample2 = (x2 @ w) + b1

    Sample 3: y1_sample3 = (x3 @ w) + b1

There is only ONE b1 stored in memory. It was broadcast (copied) into every single row.

Because that single b1 touched all 3 samples:

    When you change b1, sample 1's error changes.

    When you change b1, sample 2's error changes.

    When you change b1, sample 3's error changes.

So by the multivariable chain rule, to find how much Total Loss changes when you tweak b1:

dL / db1 = (dL / dy1_sample1) + (dL / dy1_sample2) + (dL / dy1_sample3)

dL / db1 = g1_sample1 + g1_sample2 + g1_sample3

You just sum down column 0.

Concrete Numbers

Say your batch size is 3, and your grad_out has 2 output neurons:

Python

grad_out = torch.tensor([

    [ 2.0, -1.0],  # row 0 (Sample 1): g1 = 2.0,  g2 = -1.0

    [ 1.5,  0.5],  # row 1 (Sample 2): g1 = 1.5,  g2 =  0.5

    [-0.5,  2.0]   # row 2 (Sample 3): g1 = -0.5, g2 =  2.0

])

Look at the columns:

    Column 0 (Neuron 1): 2.0 + 1.5 + (-0.5) = 3.0

    Column 1 (Neuron 2): -1.0 + 0.5 + 2.0 = 1.5

Final bias gradient:

grad_bias = [3.0, 1.5]

In code, summing down the rows (dimension 0) does this exact operation:

A simple shortcut to find without this much mess

grad_inputs -> shape(B,in_features)  and we have the grad_out(B,out) and weights(out,in)

so try to figure out what combinations can give the shape of the grad_inputs

grad_inputs = grad_out @ weights(out,in)

grad_weights -> shape(out,in) and we have the grad_out(B,out) and inputs(B,in) 

so we either need the grad_out.T @ inputs

as we sum up the gradients ,we have discussed above

grad_bias = grad_out.sum(dim=0)

now we need to understand about the relu

so as we know the relu ,how does it kills the negative values to 0

and now we need to assume a simple eqn

y = R(x)

and we wanted to find the loss

dL/dx = dL/dy * dy/dx (chain rule)

so we can get the dL/dx = grad_out * dy/dx

as this have only two cases 

1)if x >0 then x 

2)0 as the x <0 

and derivate is 0 for the 0

so the backward for the relu is 

grad_out * (self.x>0) so those which are activated,they can only have the backprop remaining all are gonna be 0

as grad_out*0 = 0

backpropagation for the mlp layer

now the model will get the linear activation and non linear activations

we need to pass the in_features and out_features and hidden features,where there are hidden layers between the in and out layers

first we pass through the linear layer 1 and afer passing through that layers we can pass them to the non linear activations like the relu or gelu

for the linear layer 1 -> (B,in) we are gonna using the weights as the (hidden,in) same linear layer work,here the out is replaced wth the hidden because this is an intermediate layer

so they are now passed through the relu and half of them are killed and we can only leave the positive values outside 

so this is the linear + relu

so in the next layer we are gonna make the hidden_features(our new in) -> out_features

so we send these into the linear layer

this is why we did actually use the in,hidden as they will be sent as the params to the linear layer and same mechanism of the linear layer

as the mlp is like,piece of code which actually use the layers and integrates them and uses the non linear activations and even dropouts

basically this is a bridge between the input neurons to the final layer

backpropagation in this mlp layer is 

linear1->relu->linear2->output

so back prop is,reverse ,we will give the grad_out and

the gradients are passed in reverse from the layer 2 to the inputs

so the gradients_out are passed int the layer 2 and they will passed back into the relu

and gradients from the relu are passed back to the linear1 and we will return that final values

mse means mean squared error

this is like the findig the distance between the two points ... so this says the how much does the model's predictions are from the actual true outputs and at the end we use the

mean of those values ,as we can see below why mean is consistent

batch size 2:

Pred: [50, 60]

True: [100, 80]

Errors: [-50, -20]

Squared: [2500, 400]

Sum = 2900

Mean = 2900/2 = 1450

batch size 4:

Pred: [50, 60, 70, 80]

True: [100, 80, 90, 70]

Errors: [-50, -20, -20, 10]

Squared: [2500, 400, 400, 100]

Sum = 3400

Mean = 3400/4 = 850

so the second model performs better because,if we did take the mean we can say how much does the model gives the errors from the whole group

because the loss changes with the size,so we can't compare fairly and mean can assure us to compare them fairly

for the back propogation we just derivate it and the power 2 comes as the coefficient

for the training steps

the basic concept is 

1)first what are the predictions from the model and we can know,it is the forward pass

2)what is the loss ?we can kow it ,but using the mse loss forward

3)now we need to use the backprop,as we did the find this from the ,mse loss backward so that,we can find the loss from the individual,as the loss from entire network

we can now find the loss and we will send that loss to the layers and we use the mlp backwrd

In [ ]:


class LinearLayer:
    def __init__(self,in_features,out_features):
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.bias = torch.zeros(out_features)

    def forward(self,x):
        self.x  = x
        return x @ self.weights.T+self.bias

    def backward(self,grad_out):
        grad_inputs = grad_out @ self.weights
        x_flat = self.x.reshape(-1, self.x.shape[-1])
        grad_out_flat = grad_out.reshape(-1, grad_out.shape[-1])
        self.grad_weights = grad_out_flat.T @ x_flat
        self.grad_bias = grad_out_flat.sum(dim=0)

        return grad_inputs

        
class Relu:
    def forward(self,x):
        self.x = x
        return torch.clamp(x,min=0)

    def backward(self,grad_out):
        return grad_out * (self.x > 0)


        
class MyMlp:
    def __init__(self,in_features,hidden_features,out_features):
        self.linear1 = LinearLayer(in_features,hidden_features)
        self.relu = Relu()
        self.linear2 = LinearLayer(hidden_features,out_features)

    def forward(self,x):
        x = self.linear1.forward(x)
        x = self.relu.forward(x)
        x = self.linear2.forward(x)

        return x

    def backward(self,grad_out):
        grad_linear2 = self.linear2.backward(grad_out)
        grad_relu   = self.relu.backward(grad_linear2)
        grad_linear1 = self.linear1.backward(grad_relu)
        
        return grad_linear1
        
        
class MyMseLoss:

    def forward(self,y_pred,y_true):
        self.y_pred = y_pred
        self.y_true= y_true
        loss = torch.mean((y_pred - y_true)**2)
        return loss

    def backward(self):
        N = self.y_pred.numel()

        grad_out = (2/N)*(self.y_pred-self.y_true)

        return grad_out


X = torch.randn(32,1000)
Y = torch.randn(32,10)
in_features = 1000
hidden_features = 64
out_features = 10
lr = 0.0001

model = MyMlp(in_features,hidden_features,out_features)
loss_fn = MyMseLoss()

for epoch in range(100):

    # forward pass
    y_pred = model.forward(X)

    # loss 
    loss = loss_fn.forward(y_pred,Y)

    # back prop
    grad_loss = loss_fn.backward()
    model.backward(grad_loss)


    with torch.no_grad():
        # update lauer 1
        model.linear1.weights -= lr*model.linear1.grad_weights
        model.linear1.bias    -= lr * model.linear1.grad_bias

        # Update Layer 2
        model.linear2.weights -= lr * model.linear2.grad_weights
        model.linear2.bias    -= lr * model.linear2.grad_bias


    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}")
